In [31]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# خواندن همه فایل‌های parquet در مسیر مشخص
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/012026/Data/MEDS_MDPS/data/held_out/*.parquet')]    #     Zahra/Data-07-2025/MDP/MEDS_811/data/train
)

# تبدیل به pandas DataFrame
df = dataset.to_pandas_dataframe()
df.head(15)


{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,subject_id,time,code,numeric_value
0,27,NaT,GENDER//Kvinde,NaN
1,27,1957-05-14 00:00:00,DOB,NaN
2,27,2009-01-01 00:00:00,D/DE052A,NaN
3,27,2017-12-11 15:15:00,P/DZ123AA,NaN
4,27,2017-12-11 15:15:00,P/UXRC45,NaN
5,27,2018-02-11 00:00:00,D/DK920,NaN
6,27,2018-02-11 00:00:00,D/DA419,NaN
7,27,2018-02-11 00:00:00,D/DG039,NaN
8,27,2018-02-11 00:00:00,D/DE876,NaN
9,27,2018-02-11 10:49:00,ADMISSION_ADT,NaN


In [32]:
len(df)

55985081

In [33]:
import pandas as pd

# فرض بر این که فایل CSV رو داری
# df = pd.read_csv("your_file.csv")  # یا مستقیم اگر DataFrame آماده‌ست، نیازی نیست

# دسته‌بندی هر subject_id بر اساس وجود کدهای مختلف
m_patients = df[df['code'].str.startswith('M/', na=False)]['subject_id'].unique()
p_patients = df[df['code'].str.startswith('P/', na=False)]['subject_id'].unique()
d_patients = df[df['code'].str.startswith('D/', na=False)]['subject_id'].unique()
s_patients = df[df['code'].str.startswith('S/', na=False)]['subject_id'].unique()

# کل بیماران منحصربه‌فرد
all_patients = df['subject_id'].unique()

# نمایش آمار
print(f"Whole Patients: {len(all_patients)}")
print(f"The patients has M-medication Codes: {len(m_patients)}")
print(f"The patients has D-diagnosis Codes: {len(d_patients)}")
print(f"The patients has P-Procedure Codes: {len(p_patients)}")
print(f"The patients has S-SKS Codes: {len(s_patients)}")


Whole Patients: 221803
The patients has M-medication Codes: 151356
The patients has D-diagnosis Codes: 221749
The patients has P-Procedure Codes: 212660
The patients has S-SKS Codes: 53203


In [34]:
subject_counts = df['subject_id'].value_counts()


In [35]:
subject_counts

106        63791
1154761    48106
1011632    45271
1580453    44345
926005     43380
           ...  
1484041        3
1826325        3
1682596        2
85775          2
1506679        2
Name: subject_id, Length: 221803, dtype: int64

In [36]:
p_Num = df[df['code'].str.startswith('S/', na=False)]

In [37]:
p_Num

,subject_id,time,code,numeric_value
3,27,2017-12-11 15:15:00,P/DZ123AA,NaN
4,27,2017-12-11 15:15:00,P/UXRC45,NaN
17,27,2018-02-11 12:32:00,P/KTAB00,NaN
26,27,2018-02-11 12:51:00,P/UXCA00,NaN
27,27,2018-02-11 12:51:00,P/UXRC00,NaN
...,...,...,...,...
55985076,2217923,2023-10-11 13:56:00,P/AAF22,NaN
55985077,2217923,2023-10-11 13:56:00,P/BOHJ18A3,NaN
55985078,2217923,2024-01-11 13:32:00,P/BOHJ18A3,NaN
55985079,2217923,2024-04-05 09:54:00,P/AAF22,NaN


In [38]:
import pandas as pd

# پیدا کردن سطرهایی که فقط codeهای نوع /P دارن
only_p = df[df['code'].str.startswith('S/', na=False)]

# بیماران با فقط /P کد
subject_ids_only_p = only_p['subject_id'].unique()

# حالا بیماران با codeهای غیر از /P
not_p = df[~df['code'].str.startswith('S/', na=False)]
subject_ids_with_non_p = set(not_p['subject_id'].unique())

# حذف بیمارانی که فقط /P دارن
only_p_ids_to_exclude = [sid for sid in subject_ids_only_p if sid not in subject_ids_with_non_p]

print("Number of patients with only Surgery code: ", only_p_ids_to_exclude)


In [ ]:
df_filtered = df[~df['code'].str.startswith('S/', na=False)]

In [ ]:
subject_counts_MDS = df_filtered['subject_id'].value_counts()

In [ ]:
subject_counts_MDS

In [ ]:
subject_counts_df = subject_counts.reset_index()
subject_counts_df.columns = ['subject_id', 'original_count']

subject_counts_MDS_df = subject_counts_MDS.reset_index()
subject_counts_MDS_df.columns = ['subject_id', 'new_count']


In [ ]:
import pandas as pd
comparison_df = pd.merge(subject_counts_df, subject_counts_MDS_df, on='subject_id', how='outer')


In [ ]:
comparison_df['difference'] =  comparison_df['original_count'] - comparison_df['new_count']


In [ ]:
comparison_df = comparison_df.sort_values(by='difference', ascending=False)


In [ ]:
comparison_df

In [ ]:
unchanged_count = (comparison_df['difference'] == 0).sum()
print("unchanged_count", unchanged_count)


In [ ]:
comparison_df['abs_diff'] = comparison_df['difference'].abs()
most_changed = comparison_df.sort_values(by='abs_diff', ascending=False)


In [ ]:
print(most_changed.head(10))


In [ ]:
changed_df = comparison_df[comparison_df['difference'] != 0]
min_new_count = changed_df['new_count'].min()
lowest_new_count_patients = changed_df[changed_df['new_count'] == min_new_count]


In [ ]:
lowest_new_count_patients = lowest_new_count_patients.rename(
    columns={
        'original_count': 'MDPS codes',
        'new_count': 'MDP codes'
    }
)


In [ ]:
lowest_new_count_patients

.str.upper() شرط را case-insensitive می‌کند

بل از فیلتر، ستون را به pd.StringDtype() تبدیل می‌کند؛ این کار رفتار .str را پایدار و قابل پیش‌بینی می‌کند (<NA> به‌جای NaN)



In [ ]:
# کل بیماران منحصربه‌فرد
all_patients = df_filtered['subject_id'].unique()
len(all_patients)
# نمایش آمار

In [ ]:
len(df_filtered)

In [ ]:
import pandas as pd
import numpy as np

# امن‌تر: اگر code نال یا غیررشته‌ای بود اذیت نکنه
df['code'] = df['code'].astype('string')
df_filtered = df[~df['code'].str.upper().str.startswith('S/', na=False)].copy()
print("kept rows MDP:", len(df_filtered), " / total:", len(df))


In [ ]:
import pandas as pd
import numpy as np

# اطمینان از نوع‌ها (برای خروجی تمیز و بدون خطا)
df_filtered = df_filtered.copy()
df_filtered['subject_id']    = pd.to_numeric(df_filtered['subject_id'], errors='coerce').astype('Int64')
df_filtered['numeric_value'] = pd.to_numeric(df_filtered['numeric_value'], errors='coerce').astype('float32')
df_filtered['time']          = pd.to_datetime(df_filtered['time'], errors='coerce', utc=False)

# ردیف‌های بدون subject_id را حذف کنیم (نمی‌توان شارد کرد)
df_filtered = df_filtered.dropna(subset=['subject_id']).copy()
df_filtered['subject_id'] = df_filtered['subject_id'].astype('int64')

# ۳۶ شارد: هر بیمار فقط در یک فایل (mod 36)
N_SHARDS = 45
df_filtered['__shard__'] = (df_filtered['subject_id'] % N_SHARDS).astype('int16')

print("rows to write:", len(df_filtered))


In [ ]:
import numpy as np
import os

N_SHARDS = 5   #45 for Whole # 36 when we have split
OUT_DIR = "./_held_outMDPS_withoutS_sharded"
os.makedirs(OUT_DIR, exist_ok=True)

cols_out = ['subject_id', 'time', 'code', 'numeric_value']

# فرض: subject_id قبلاً int64 شده و NaNها حذف شده‌اند (طبق سلول قبلی‌ات)
sid_mod = (df_filtered['subject_id'].to_numpy(dtype=np.int64, copy=False) % N_SHARDS)

written = 0
for k in range(N_SHARDS):
    mask = (sid_mod == k)                 # بدون ستون اضافی، فقط یک آرایهٔ NumPy
    part = df_filtered.loc[mask, cols_out].sort_values(['subject_id','time'])
    # اگر می‌خوای حتماً ۳۶ فایل 0..35 داشته باشی حتی اگه خالی باشن:
    # if part.empty:
    #     part = part.iloc[0:0]  # فایل صفر-سطر با همان ستون‌ها
    part.to_parquet(os.path.join(OUT_DIR, f"{k}.parquet"),
                    engine="pyarrow", compression="snappy", index=False)
    written += 1

print(f"Done. wrote {written} parquet files into {OUT_DIR}")


In [ ]:
import pyarrow.parquet as pq

OUT_DIR = "./_held_outMDPS_withoutS_sharded"

def parquet_num_rows(path):
    pf = pq.ParquetFile(path)
    md = pf.metadata
    return sum(md.row_group(i).num_rows for i in range(md.num_row_groups))

total = 0
for f in sorted(p for p in os.listdir(OUT_DIR) if p.endswith('.parquet')):
    n = parquet_num_rows(os.path.join(OUT_DIR, f))
    total += n
    print(f, "rows:", n)
print("TOTAL rows:", total)


In [ ]:
from azureml.data.datapath import DataPath
from azureml.data.dataset_factory import FileDatasetFactory
import os

OUT_DIR = "./_held_outMDPS_withoutئ_sharded"
DST_PREFIX = "Zahra/012026/Data/MEDS_MDP/data/held_out"  #held_out"  # بدون اسلشِ اول
''

# اطمینان: پوشه خروجی وجود دارد و فایل parquet داخلش هست
print("Local files to upload:", len([f for f in os.listdir(OUT_DIR) if f.endswith(".parquet")]))

# مقصد روی Datastore
target = DataPath(datastore, DST_PREFIX)

# آپلود همه محتویات OUT_DIR به DST_PREFIX
_ = FileDatasetFactory.upload_directory(
    src_dir=OUT_DIR,
    target=target,
    overwrite=True,
    show_progress=True,
)
print(f"Uploaded to datastore path: {DST_PREFIX}")


In [ ]:
paths = Dataset.File.from_files(path=[(datastore, f"{DST_PREFIX}/*.parquet")]).to_path()
print("Found in datastore:", len(paths))
print(paths[:20])
